# Pathways that only come out at 200M reads

Exon skipping events detected at 50M, 100M, 150M and 200M reads are passed to
NEASE, and the pathways that become significant at 200M but are not
significant at 150M are reported. NEASE scores the events against its own
protein-interaction background; the input is restricted to genes detected in
all 100 subsamples at that depth.

The enrichment tables in `data/derived/nease/` hold one NEASE run per sample
and depth, scored on the DICAST-unified events.

In [1]:
from pathlib import Path

import pandas as pd

# Locate the top of the clone, so the notebook runs from any directory inside it.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "asdepth").is_dir()), None)
if ROOT is None:
    raise RuntimeError("run this from inside a clone of RNA-Seq_Depth_for_AS")
DATA = ROOT / "data" / "derived" / "nease"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

DEPTHS = [50, 100, 150, 200]

# The three datasets reported here.
SAMPLES = {
    "adipose_1": "Adipose (pre-treatment)",
    "adipose_2": "Adipose (post-treatment)",
    "hypothalamus": "Hypothalamus",
}

# Depth to compare the deepest run against.
PREVIOUS_DEPTH = 150

## Pathways gained at 200M

In [2]:
rows = []
for sample, title in SAMPLES.items():
    results = {
        depth: pd.read_csv(DATA / f"{sample}_{depth}M_nease", sep="\t")
        for depth in DEPTHS
    }
    significant = {
        depth: set(table[table["adj p_value"] < 0.05]["Pathway name"])
        for depth, table in results.items()
    }

    gained = significant[200] - significant[PREVIOUS_DEPTH]

    deepest = results[200]
    for _, row in deepest[deepest["Pathway name"].isin(gained)].iterrows():
        rows.append(
            {
                "dataset": title,
                "KEGG pathway": row["Pathway name"].replace(
                    " - Homo sapiens (human)", ""),
                "p-adjusted": row["adj p_value"],
            }
        )

table_s7 = (
    pd.DataFrame(rows)
    .sort_values(["dataset", "p-adjusted"])
    .reset_index(drop=True)
)
table_s7.to_csv(RESULTS / "Table_S7.tsv", sep="\t", index=False)

print(table_s7["dataset"].value_counts().to_string())
table_s7

dataset
Adipose (pre-treatment)     21
Hypothalamus                21
Adipose (post-treatment)    11


,dataset,KEGG pathway,p-adjusted
0,Adipose (post-treatment),SNARE interactions in vesicular transport,0.005566
1,Adipose (post-treatment),NOD-like receptor signaling pathway,0.005925
2,Adipose (post-treatment),Herpes simplex infection,0.010257
3,Adipose (post-treatment),Epstein-Barr virus infection,0.010279
4,Adipose (post-treatment),TNF signaling pathway,0.010279
5,Adipose (post-treatment),Toll-like receptor signaling pathway,0.021010
6,Adipose (post-treatment),Measles,0.024154
7,Adipose (post-treatment),Antigen processing and presentation,0.026103
8,Adipose (post-treatment),Endocytosis,0.028136
9,Adipose (post-treatment),mTOR signaling pathway,0.030602
